# **Welcome to your first Bioinformatics Lab!**

Right now you are in the lab working with the transcription factor TBP. You have one week to scratch the surface of cloning TBP.

The work of a real-life researcher does not end after investigating their target transcription factor for one week in the lab. Instead, many researchers relies on using bioinformatic tools to learn much more about their target.

This exercise shows what a real-life researcher might do to understand the mechanism of the transcription factor TBP even more. You will work with a real dataset from a ChIP-seq experiment investigating which sites on the humane genome TBP binds.

## How to complete this notebook

Throughout the exercise, you will answer 7 questions. The right answer will give you letters, numbers or symbols. **Note these down as you go and complete the exercise by entering the correct code at the end.**



**Question 1:** You have been handed out a paper with components of a full figure illustrating the ChIP-seq workflow. However the headlines and the order have been scrambled. Please cut out each step and sort them in the correct order. **Note down the purple letters in the order you created**.  


# **How to run a Jupyter Notebook**

> 🟧 Jupyter Notebooks
>
> This interface is called *Jupyter Notebook* (from now on notebook).
In this case, it is served within the free tool *Google Collab*. Inside this notebook you can write code and text, which makes it perfect for sharing and reporting data analyses. It also allows to download data, insert images or install packages, all within the notebook.
You may see a panel to the right called "Release Notes". You can close this, so you only look at the notebook.

> How to use this notebook?
>
> - You can run the code in each cell by clicking on the "Run cell" sign. This sign can be found by hovering your mouse cursor over the programming cell you want to run. The sign is shaped like a Play-icon. When the code finished running, a small green check sign will appear on the top left side.
>
> - You need to run the cells in **sequential order**, please do not run a cell until the one above is finished running and do not skip any cells.
>
> - Each cell contains a short description of the code and the output you should get. Please try not to focus on understanding what each command line does, as this is not the purpose of the exercise


### **Hands-on!! First, let's setup our notebook!**
*   Running the cell below will install all the necessary programs that you will  be using. This might take some time, approximately 10 min. (Please ignore all the messages you see when installing the various packages).

In [ ]:
import os

def run(cmd, msg):
    print(msg)
    os.system(cmd + " > /dev/null 2>&1")

# Install system tools
run("sudo apt -y install bowtie2 samtools bedtools", "Installing bowtie2, samtools, and bedtools...")
run("pip install macs3", "Installing macs3...")

# Install Miniconda
run("wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh", "Downloading Miniconda...")
run("bash Miniconda3-latest-Linux-x86_64.sh -b -p /content/miniconda", "Installing Miniconda...")
run("rm Miniconda3-latest-Linux-x86_64.sh", "Cleaning up...")

# Add Conda channels + accept ToS
run("/content/miniconda/bin/conda config --add channels bioconda", "Adding bioconda channel...")
run("/content/miniconda/bin/conda config --add channels conda-forge", "Adding conda-forge channel...")
run("/content/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main", "Accepting ToS for main...")
run("/content/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r", "Accepting ToS for r...")

# Install mamba
run("/content/miniconda/bin/conda install mamba -n base -c conda-forge -y -q", "Installing mamba...")

# Make sure PATH includes miniconda
os.environ['PATH'] = "/content/miniconda/bin:" + os.environ['PATH']

# Create R 4.3 environment
run("mamba create -n myenv_r43 -c conda-forge -c bioconda "
    "r-base=4.3 bioconductor-chipseeker "
    "bioconductor-txdb.hsapiens.ucsc.hg38.knownGene r-ggplot2 "
    "-y -q", "Creating R 4.3 environment with ChIPseeker & TxDb...")

print("✅ Setup complete! Your environment is ready.")


Installing bowtie2, samtools, and bedtools...
Installing macs3...


# **Data analysis**
You and your fellow researchers have performed the TBP ChIP-seq experiment on the human Chromosome 1. You now have all the sequencing data and are ready to analyse it.

As you saw on the last step in the ChIP-seq figure, the data analysis starts with aligning the reads to the reference genome, in this case the human chromosome 1, and then finding potential binding sites on the genome.

The code cell below will import the read data from github and then start the data analysis. Run the code cell now to see your first result.

In [ ]:
###-------The following code imports the data from github---#
!rm -r sample_data/
!rm -rf data
!git clone https://github.com/MolXLab/MBII.git data

###------The following code performs read alignment and organizes the results---#
# Read alignment
!mkdir -p bowtie_index
!bowtie2-build /content/data/NC_000001.11_subset.fasta bowtie_index/Chr1_index > /dev/null
!mkdir -p bowtie2_results
!bowtie2 -q --local \
-x  bowtie_index/Chr1_index \
-U /content/data/TBP_chr1_subset.fastq \
-S bowtie2_results/TBP_chr1.sam &> bowtie2_results/TBP_chr1.log

# Compress SAM output
!samtools view -q 10 -u bowtie2_results/TBP_chr1.sam | samtools sort -o bowtie2_results/TBP_chr1_sorted.bam
!samtools index bowtie2_results/TBP_chr1_sorted.bam

###------- The following code performs peak calling------###
!mkdir -p macs3_results
!macs3 callpeak -t /content/bowtie2_results/TBP_chr1_sorted.bam -f BAM -g 10000000 --outdir /content/macs3_results/ -m 1 100 -n TBP --verbose 0

!wget -q -O code.txt https://raw.githubusercontent.com/MolXLab/MBII/main/code.txt
with open('code.txt', 'r') as file:
    correct_code = file.read().strip()  # Reads the file and removes any extra whitespace/newlines


print("Done!!")

In [ ]:
!/content/miniconda/bin/conda run -n myenv_r43 Rscript /content/data/RScript.R

In [ ]:
from IPython.display import Image, display

x = Image("/content/covplot_TBP.png", height=200, width=800)
display(x)

### **Peak analysis**
The code above takes all the sequenced reads as input and aligns them to the genome, matching the sequence of each read to the corresponding sequence in the genomic DNA. The number of reads binding to specific regions of the genome is then quantified and visualized as peaks in a graph. Each cluster of peaks represents a potential binding site for TBP.

Use the graph to answer the following questions:

**Question 2**: What does a high peak at a binding site typically indicate compared to a low peak? Note the symbol that corresponds to the correct option below.

%) A high peak indicates that fewer reads were aligned to the binding site, suggesting weaker binding affinity of the protein at that site.

!) A high peak suggests a strong enrichment or binding affinity at that site, meaning more sequencing reads aligned to this region.

?) A high peak represents a control region where the protein does not bind, while a low peak represents an area of high binding affinity.

") There is no difference between high and low peaks; all peaks represent equal binding affinity.


**Question 3**: How many regions have a peak intensity of more than 150?

### Analysis of TBP binding sites

The data obtained from the peak analysis allows the real-life scientist to investigate the types of genomic regions where TBP binds. They have quantified and visualized these results in the annotation bar plot shown above. Load the plot by running the code below.


In [ ]:
x= Image("/content/plotAnnoBar.png", width=700, height=700) ; display(x)


**Question 4**: Which region is found **closest to the start** of a gene and helps initiate transcription?

   A) Promoter (<=1kb)

   B) 3' UTR

   C) Other Exon

**Question 5**: Which region is typically found **far from any known genes** and may or may not be involved in gene expression?

   A) 1st Exon

   B) Promoter (2-3kb)

   C) Distal Intergenic

**Question 6**: Which region is located **after the coding sequence** of a gene and can impact mRNA stability?

   A) Promoter (1-2kb)

   B) 3' UTR

   C) 1st Intron


**Question 7**: What kind of genomic element does TBP mostly bind to?

   A) Promoter (<=1kb)

   B) Downstream (<=300)

   C) Other Exon

You have now answered all 7 questions. Run the next codecell and check to see if your answers are correct.

**Remember to write uppercase letters!!**

In [ ]:
from IPython.display import HTML, display
# Final cell to check answer
entered_code = input("Enter the final code to see if you got it right: ")

# Check if the entered code is correct
if entered_code == correct_code:
    display(HTML("<h1 style='color: green;'>🎉 Correct! Great job! 🎉</h1>"))
    display(HTML("<img src='https://media.giphy.com/media/X9izlczKyCpmCSZu0l/giphy.gif' width='300'>"))  # Celebration GIF
else:
    display(HTML("<h1 style='color: red;'>Incorrect. Try again!</h1>"))
    display(HTML("<img src='https://media.giphy.com/media/CoND5j6Bn1QZUgm1xX/giphy.gif' width='300'>"))